# 01. Data Diagnosis

## Goal

원본 스내치 영상에서 `videos.csv` Manifest를 안전하게 생성하거나 불러오고, 다음 분석 단계 전에 기본 데이터 상태를 확인합니다.

## Setup

이 노트북은 프로젝트 루트 또는 `notebooks/` 디렉터리에서 실행할 수 있습니다. 재사용 로직은 `metadata.py`에서 가져옵니다.

In [ ]:
from pathlib import Path
import os

import pandas as pd

from snatch_analysis.metadata import write_video_manifest


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("프로젝트 루트 또는 notebooks 디렉터리에서 실행해 주세요.")

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

## Steps

### 1. 경로와 재생성 여부 설정

`REBUILD_MANIFEST=False`가 기본값입니다. 기존 CSV가 있으면 그대로 불러오므로 이후 직접 기록할 `has_plate`, `camera_motion` 값이 유지됩니다. 원본 영상이 추가되었거나 자동 메타데이터를 처음부터 다시 만들 때만 `True`로 바꾸세요.

In [ ]:
VIDEO_DIR = Path("data/raw/videos")
MANIFEST_PATH = Path("data/metadata/videos.csv")
REBUILD_MANIFEST = False

### 2. Manifest 생성 또는 불러오기

CSV가 없거나 `REBUILD_MANIFEST=True`이면 `write_video_manifest()`가 영상을 읽어 새 CSV를 만듭니다. 그 외에는 기존 CSV를 불러옵니다.

In [ ]:
if REBUILD_MANIFEST or not MANIFEST_PATH.exists():
    manifest = write_video_manifest(VIDEO_DIR, MANIFEST_PATH)
    manifest_action = "생성"
else:
    manifest = pd.read_csv(MANIFEST_PATH)
    manifest_action = "기존 파일 불러오기"

print(f"Manifest: {manifest_action}")
print(f"Path: {MANIFEST_PATH}")

## Checks

원본 MP4 개수와 Manifest 행 수, 중복 ID, 영상 읽기 상태를 확인합니다. 문제가 있으면 다음 단계로 넘어가지 않고 여기서 원인을 확인합니다.

In [ ]:
video_count = len(list(VIDEO_DIR.glob("*.mp4")))
if video_count == 0:
    raise FileNotFoundError(f"MP4 파일이 없습니다: {VIDEO_DIR}")
if len(manifest) != video_count:
    raise ValueError(f"영상 {video_count}개와 Manifest {len(manifest)}행이 일치하지 않습니다.")
if manifest["video_id"].duplicated().any():
    raise ValueError("중복 video_id가 있습니다.")

diagnosis_summary = pd.Series(
    {
        "video_files": video_count,
        "manifest_rows": len(manifest),
        "unique_people": manifest["person_id"].nunique(),
        "read_ok": int((manifest["read_status"] == "ok").sum()),
        "read_error": int((manifest["read_status"] != "ok").sum()),
    },
    name="count",
)
diagnosis_summary.to_frame()

### 3. Manifest 일부 확인

출력이 너무 길어지지 않도록 앞의 5개 행만 확인합니다.

In [ ]:
manifest.head(5)

## Next Steps

다음 작업에서는 Manifest 분포를 시각화하고 대표 프레임을 확인한 뒤 `has_plate`와 `camera_motion`을 기록합니다. 수동 기록 이후에는 `REBUILD_MANIFEST=False`를 유지합니다.